# 🔬 SynthID Remover — Google Colab

HF Space용 `app.py`를 Colab에서 그대로 돌리는 노트북입니다.
ComfyUI v0.7.0 + 커스텀 노드팩 + 모델 28.1 GiB를 받아서 Gradio UI(공개 링크)로 실행합니다.

> ⚠️ **하드웨어 최소 요건** — Qwen-Image 2512(20B)를 4스텝 돌립니다.
> - **Colab 무료(T4 16GB)**는 메모리가 부족해 OOM 가능성이 높습니다. 저메모리 모드로 시도는 되지만 작은 이미지를 쓰세요.
> - **Colab Pro+ (A100 40GB, 또는 L4 24GB + 고RAM)** 를 권장합니다.

> 📦 **로컬 수정사항 반영**: 이 노트북은 아래 `REPO_URL`에서 코드를 clone합니다.
> 로컬에서 바꾼 내용(RGBA 지원, 저메모리 모드 등)을 먼저 push한 뒤 실행하세요.

In [ ]:
import os, shutil, subprocess

print('=== GPU ===')
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True,
)
print(r.stdout.strip() or 'nvidia-smi not found')

def ram_gb():
    try:
        return round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3, 1)
    except Exception:
        return None

print(f'System RAM : {ram_gb()} GB')
print(f'Disk free  : {round(shutil.disk_usage("/").free / 1024**3, 1)} GB')

## 예상 메모리 사용량

| 스택 | 파일 | 가중치 |
| --- | --- | ---: |
| Qwen-Image | unet Q4_K_M | 12.39 GiB |
| Qwen-Image | CLIP Q4_K_M | 4.36 GiB |
| Qwen-Image | ControlNet | 2.11 GiB |
| Qwen-Image | Lightning LoRA fp32 | 1.58 GiB |
| Qwen-Image | VAE | 0.24 GiB |
| **Qwen 합계** | | **20.68 GiB** |
| Z-Image | unet Q4_K_M | 4.64 GiB |
| Z-Image | CLIP IQ4_XS | 2.11 GiB |
| Z-Image | VAE | 0.31 GiB |
| **Z-Image 합계** | | **7.06 GiB** |
| 디텍터 | YOLOv8n + SAM | 0.36 GiB |
| **전체 가중치** | | **28.10 GiB** |

활성화(런타임) ~4–8 GiB + PyTorch/CUDA 오버헤드 ~2–3 GiB 추가.

- **저메모리 모드(스택 교체)**: 피크 ≈ Qwen 20.68 + 활성화 ~5 + 오버헤드 ~2 ≈ **27–29 GiB**
- **상주 모드(두 스택 동시)**: 28.1 + 활성화 ~6 + 오버헤드 ~2 ≈ **35–39 GiB**

→ T4(16GB VRAM + ~13GB RAM = ~29GB 합산)는 저메모리 모드로도 한계치에 걸려 불안정.
→ A100 40GB 또는 L4 24GB + 고RAM에서 안정적으로 동작.

## 1. 코드 clone + 의존성 설치

`REPO_URL`을 본인 저장소로 바꾸세요.

In [ ]:
REPO_URL = 'https://huggingface.co/spaces/dennny123/synth-id-remover'
WORKDIR = '/content/synthid-remover'

%cd /content
!rm -rf {WORKDIR}
!git clone {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!pip install -q -r requirements.txt

## 2. 설치 & 모델 다운로드 (첫 실행은 오래 걸림)

아래 셀에서 `import app`이 ComfyUI 클론 + 모델 28.1 GiB 다운로드 + 노드 부트스트랩을 전부 수행합니다.

In [ ]:
import os, sys
os.environ['SYNTHID_LOW_MEMORY'] = '1'   # Colab T4: 스택 교체로 메모리 절약
sys.path.insert(0, '/content/synthid-remover')

import app  # noqa: E402
print('[colab] setup complete')

## 3. 실행

공개 Gradio 링크가 출력됩니다. 실패 시 Colab의 포트 포워딩(또는 ngrok)을 쓰라는 안내가 나옵니다.

In [ ]:
try:
    app.demo.queue().launch(share=True, debug=True)
except Exception as e:
    print('share=True 실패:', e)
    print('로컬 7860으로 실행 — Colab 우측 하단 포트 포워딩 또는 ngrok 사용')
    app.demo.queue().launch(server_name='0.0.0.0', server_port=7860, debug=True)